# Aula 1 - LLM APIs

## Template da V1 do projeto

Neste notebook, você vai montar a primeira versão do assistente de estudos da monitoria.

As células com `TODO` são partes que você precisa completar. Não coloque sua API key diretamente no notebook, utilize o `.env`.

## Objetivos

Ao final, você deve conseguir:

- configurar um cliente para a API da Groq;
- montar mensagens com `system` e `user`;
- enviar uma requisição e ler a resposta;
- observar o uso de tokens;
- explicar por que chamadas separadas não têm memória automática;
- testar o efeito da temperatura;
- salvar a V1 em um commit.

## 1. Configuração do ambiente

Antes de executar o notebook:

1. crie um ambiente virtual;
2. instale as dependências de `requirements.txt`;
3. copie `env.example` para `.env`;
4. preencha `GROQ_API_KEY` no `.env`.

In [3]:
# Passo a passo para criar o .venv: 
# Abra o Terminal na raiz do projeto e digite:
# python -m venv venv
# Depois digite: 
# cd "Aula 01"
# pip install -r requirements.txt

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Monte as mensagens

A aplicação controla o comportamento do assistente usando mensagens.

Use uma mensagem `system` para definir o papel do assistente e uma mensagem `user` para fazer uma pergunta sobre a monitoria.

In [4]:
# TODO: crie uma lista chamada `messages` com uma mensagem `system` e uma mensagem `user`.
messages = [
    {
        "role": "system",
        "content": "Você é um amigo próximo e prestativo que vai responder às perguntas do usuário da melhor forma possível."
    },
    {
        "role": "user",
        "content": "Qual a chance de um ser humano ganhar de um avestruz numa briga um contra um?"
    }
]

## 3. Faça a primeira chamada

Envie `messages` para o modelo configurado em `MODEL` e salve a resposta em uma variável chamada `response`.

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
)

## 4. Leia o texto da resposta

A resposta contém uma lista de escolhas. Nesta atividade, vamos usar o texto da primeira escolha.

In [6]:

print(response.choices[0].message.content)

**Resumo rápido:**  
A probabilidade de um ser humano vencer um avestruz em uma briga corpo‑a‑corpo é extremamente baixa – algo em torno de **5 % ou menos**, e isso só em situações muito específicas (por exemplo, se o humano estiver armado, treinado em artes marciais e conseguir evitar os ataques do animal). Na prática, a chance de sair ileso é quase nula.

---

## Por que o avestruz tem vantagem tão grande?

| Característica | Avestruz | Ser humano (média) |
|----------------|----------|--------------------|
| **Peso** | 100 – 150 kg (machos) | 70 – 80 kg |
| **Altura** | 1,8 – 2,2 m (até 2,7 m com o pescoço) | 1,6 – 1,8 m |
| **Velocidade** | Até 70 km/h (pode manter 50 km/h por curtos períodos) | 15 – 20 km/h (corrida curta) |
| **Força de ataque** | Chute com pernas poderosas (garras que podem causar lacerações profundas) | Socos/punhos, mas força limitada |
| **Armadura natural** | Pele grossa, penas rígidas, ossos das pernas muito densos | Pele fina, ossos mais vulneráveis |
| **

## 5. Observe o uso de tokens

Além do texto, a API informa quantos tokens entraram na chamada, quantos saíram e qual foi o total.

In [7]:
# TODO: mostre `prompt_tokens`, `completion_tokens` e `total_tokens` usando `response.usage`.

print("Entrada de tokens (prompt_tokens):", response.usage.prompt_tokens)
print("Tokens de saída (completion_tokens):", response.usage.completion_tokens)
print("Total de tokens (total_tokens):", response.usage.total_tokens)

Entrada de tokens (prompt_tokens): 114
Tokens de saída (completion_tokens): 1226
Total de tokens (total_tokens): 1340


## 6. Teste a ausência de memória

Faça uma chamada dizendo que o nome do seu usuário. Depois faça uma segunda chamada perguntando o nome, mas sem reenviar a primeira mensagem.

Salve a segunda resposta em `sem_historico`. O que aconteceu?

In [8]:
# TODO: faça duas chamadas independentes e salve a segunda resposta em `sem_historico`.
client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Meu nome é daniel."}],
    temperature=0
)
sem_historico = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Qual o meu nome?"}],
    temperature=0
)

In [10]:
print(sem_historico.choices[0].message.content)

Desculpe, eu não sei qual é o seu nome. Poderia me dizer, por favor?


Agora repita a pergunta, mas envie explicitamente o histórico em `messages`. Inclua uma mensagem `assistant` representando a resposta anterior.

A aplicação cria a sensação de memória ao reenviar o contexto.

In [15]:
# TODO: crie uma nova lista de mensagens com a conversa completa e salve a resposta em `com_historico`.
com_historico = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Meu nome é daniel."},
        {"role": "user", "content": "Qual o meu nome?"},
        {"role": "assistant", "content": "Desculpe, eu não sei qual é o seu nome. Poderia me dizer, por favor?"},
    ],
    temperature=0
)
print(com_historico.choices[0].message.content)

Seu nome é Daniel.


In [16]:
# TODO: mostre o texto de `com_historico` e compare com o resultado sem histórico.
sem_historico_texto = sem_historico.choices[0].message.content
com_historico_texto = com_historico.choices[0].message.content
print("Resposta sem histórico:")
print(sem_historico_texto)
print("\nResposta com histórico:")
print(com_historico_texto)

Resposta sem histórico:
Desculpe, eu não sei qual é o seu nome. Poderia me dizer, por favor?

Resposta com histórico:
Seu nome é Daniel.


## 7. Experimente a temperatura

Use o mesmo prompt duas vezes: uma com `temperature=0` e outra com `temperature=1`.

Compare a variedade das respostas. Temperatura baixa tende a ser mais previsível; temperatura alta tende a permitir mais variação.

In [ ]:
# TODO: faça as chamadas com duas temperaturas e compare os resultados.
client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Meu nome é daniel."}],
    temperature=1
)
sem_historico = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Qual o meu nome?"}],
    temperature=1
)
print("Sem histórico com temperatura 1:")
print(sem_historico.choices[0].message.content)
com_historico = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Meu nome é daniel."},
        {"role": "user", "content": "Qual o meu nome?"},
        {"role": "assistant", "content": "Desculpe, eu não sei qual é o seu nome. Poderia me dizer, por favor?"},
    ],
    temperature=1
)
print("Com histórico com temperatura 1:")
print(com_historico.choices[0].message.content)

#Com temperatura 0, o modelo tende a ser mais direto e previsível, enquanto com temperatura 1, ele pode gerar respostas mais criativas e variadas.
#Com histórico, o modelo tem mais contexto para responder, enquanto sem histórico, ele não tem informações anteriores para basear sua resposta.

Sem histórico com temperatura 1:
Não sei o seu nome, pois você ainda não o informou. Se quiser, pode me dizer como prefere ser chamado!
Com histórico com temperatura 1:
Seu nome é Daniel.


## 8. Checkpoint da V1

Antes de fazer o commit, confirme:

- [X] a aplicação chamou a API da Groq;
- [X] a chave ficou no `.env`;
- [X] a requisição usou `system` e `user`;
- [X] a resposta apareceu;
- [X] o uso de tokens foi observado;
- [X] você comparou chamadas sem histórico e com histórico;
- [X] você testou duas temperaturas.
